# Làm sạch Dữ liệu & Xử lý Rò rỉ Dữ liệu (Data Cleaning & Leakage Resolution)

Thực hiện bước tiền xử lý và làm sạch dữ liệu thô từ thư mục `data/raw/` và lưu kết quả sạch vào `data/processed/`.

**Mục tiêu chính:**
1. Loại bỏ các ký tự thừa (`\n`, `\r`, khoảng trắng dư) trong các cột văn bản (`essay`, `prompt`, `band`).
2. Điền giá trị khuyết cho các cột lỗi từ vựng/ngữ pháp bằng chuỗi `"None"`.
3. Loại bỏ các bài viết quá dài (trên 1000 từ) trong tập Train để tránh làm quá tải mô hình và loại trừ dữ liệu rác.
4. Loại bỏ hoàn toàn các bài viết trong tập Train bị rò rỉ sang tập Test.
5. Khử trùng lặp nội bộ trong tập Train.
6. Xuất dữ liệu sạch ra các tệp `train.csv` và `test.csv` trong thư mục `data/processed/`.

In [1]:
import os
import re
import pandas as pd
import numpy as np

## 1. Thiết lập Đường dẫn Input/Output

Định nghĩa các đường dẫn tương đối để đảm bảo tính di động của mã nguồn trên các máy tính khác nhau.

In [2]:
DATA_RAW_DIR = "../data/raw"
DATA_PROCESSED_DIR = "../data/processed"

# Tạo thư mục processed nếu chưa tồn tại
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)
print(f"Thư mục lưu trữ kết quả: {os.path.abspath(DATA_PROCESSED_DIR)}")

Thư mục lưu trữ kết quả: t:\5 - Summer 2026\AES_LLM\data\processed


## 2. Tải Dữ liệu Thô và Kiểm tra Kích thước Ban đầu

In [3]:
train_raw_path = os.path.join(DATA_RAW_DIR, "dataset", "train_final.csv")
test_raw_path = os.path.join(DATA_RAW_DIR, "dataset", "test_final.csv")

df_train = pd.read_csv(train_raw_path)
df_test = pd.read_csv(test_raw_path)

print("--- KÍCH THƯỚC BAN ĐẦU ---")
print(f"df_train: {df_train.shape}")
print(f"df_test: {df_test.shape}")

--- KÍCH THƯỚC BAN ĐẦU ---
df_train: (9833, 18)
df_test: (495, 17)


## 3. Làm sạch Văn bản & Chuẩn hóa Trường dữ liệu

Chúng ta thực hiện các bước sau:
1. Loại bỏ các ký tự xuống dòng (`\n`, `\r`) và các khoảng trắng thừa.
2. Thay thế giá trị NaN trong các cột lỗi bằng chuỗi `"None"`.
3. Xóa cột `error` trống hoàn toàn trong tập train.

In [4]:
# Loại bỏ cột 'error' nếu có
if 'error' in df_train.columns:
    df_train = df_train.drop(columns=['error'])
    print("Đã loại bỏ cột 'error' trống trong df_train.")

def clean_whitespace(text):
    if pd.isna(text):
        return text
    # Thay thế ký tự xuống dòng và tab bằng khoảng trắng đơn
    text = re.sub(r'[\r\n\t]+', ' ', str(text))
    # Loại bỏ khoảng trắng trùng lặp
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Áp dụng làm sạch cho các cột văn bản chính
for df in [df_train, df_test]:
    df['essay'] = df['essay'].apply(clean_whitespace)
    df['prompt'] = df['prompt'].apply(clean_whitespace)
    # Riêng cột band, loại bỏ hoàn toàn các ký tự xuống dòng và strip
    df['band'] = df['band'].astype(str).str.strip().str.replace(r'[\r\n\t]+', '', regex=True)

# Điền giá trị rỗng cho các cột lỗi thành phần
mistake_cols = ['LR_Mistakes', 'LR_Corrections', 'GRA_Mistakes', 'GRA_Corrections']
for col in mistake_cols:
    df_train[col] = df_train[col].fillna("None").astype(str).apply(clean_whitespace)
    df_test[col] = df_test[col].fillna("None").astype(str).apply(clean_whitespace)

print("Sau khi làm sạch ký tự thừa:")
print(f"df_train: {df_train.shape}")
print(f"df_test: {df_test.shape}")

Đã loại bỏ cột 'error' trống trong df_train.
Sau khi làm sạch ký tự thừa:
df_train: (9833, 17)
df_test: (495, 17)


## 4. Loại bỏ bài viết quá dài (trên 1000 từ)

Bài viết quá dài trong IELTS Task 2 (>1000 từ) thường là dữ liệu lỗi hoặc văn bản rác, đồng thời gây quá tải ngữ cảnh mô hình (OOM). Chúng ta tiến hành loại bỏ các mẫu này khỏi tập huấn luyện.

In [5]:
MAX_WORD_LIMIT = 1000

# Tính số từ tạm thời
df_train['word_count'] = df_train['essay'].apply(lambda x: len(str(x).split()))
too_long_count = (df_train['word_count'] > MAX_WORD_LIMIT).sum()

print(f"Số lượng bài viết vượt quá {MAX_WORD_LIMIT} từ trong tập Train: {too_long_count} bài.")

# Chỉ giữ các bài viết dưới 1000 từ
df_train = df_train[df_train['word_count'] <= MAX_WORD_LIMIT].drop(columns=['word_count'])

print("Kích thước tập Train sau khi lọc bài viết quá dài:")
print(f"df_train: {df_train.shape}")

Số lượng bài viết vượt quá 1000 từ trong tập Train: 1 bài.
Kích thước tập Train sau khi lọc bài viết quá dài:
df_train: (9832, 17)


## 5. Xử lý Rò rỉ Dữ liệu (Data Leakage Resolution)

Loại bỏ tất cả các bài viết trong tập Train có nội dung trùng lặp với tập Test.

In [6]:
# Tìm các bài viết trong train xuất hiện ở test
leaked_essays = df_train['essay'].isin(df_test['essay'])
num_leaked = leaked_essays.sum()

print(f"Phát hiện {num_leaked} dòng dữ liệu bị rò rỉ (leakage) từ tập Test vào tập Train.")

# Loại bỏ các dòng bị rò rỉ
df_train = df_train[~leaked_essays]

print("\nKích thước tập dữ liệu sau khi lọc Data Leakage:")
print(f"df_train: {df_train.shape}")
print(f"df_test: {df_test.shape}")

Phát hiện 589 dòng dữ liệu bị rò rỉ (leakage) từ tập Test vào tập Train.

Kích thước tập dữ liệu sau khi lọc Data Leakage:
df_train: (9243, 17)
df_test: (495, 17)


## 6. Khử Trùng lặp Nội bộ (Internal Deduplication)

Loại bỏ các bài viết trùng lặp nội bộ trong tập Train.

In [7]:
initial_train_len = len(df_train)
df_train = df_train.drop_duplicates(subset=['essay'])
removed_train_dups = initial_train_len - len(df_train)

initial_test_len = len(df_test)
df_test = df_test.drop_duplicates(subset=['essay'])
removed_test_dups = initial_test_len - len(df_test)

print(f"Đã loại bỏ {removed_train_dups} bài viết trùng lặp nội bộ trong tập Train.")
print(f"Đã loại bỏ {removed_test_dups} bài viết trùng lặp nội bộ trong tập Test.")

print("\n--- KÍCH THƯỚC SAU KHI KHỬ TRÙNG LẶP ---")
print(f"df_train: {df_train.shape}")
print(f"df_test: {df_test.shape}")

Đã loại bỏ 954 bài viết trùng lặp nội bộ trong tập Train.
Đã loại bỏ 3 bài viết trùng lặp nội bộ trong tập Test.

--- KÍCH THƯỚC SAU KHI KHỬ TRÙNG LẶP ---
df_train: (8289, 17)
df_test: (492, 17)


## 7. Xuất File Sạch (data/processed/)

Lưu các dataframe đã được làm sạch thành các tệp tin mới.

In [8]:
train_out_path = os.path.join(DATA_PROCESSED_DIR, "train.csv")
test_out_path = os.path.join(DATA_PROCESSED_DIR, "test.csv")

df_train.to_csv(train_out_path, index=False)
df_test.to_csv(test_out_path, index=False)

print("Đã lưu các file dữ liệu sạch thành công:")
print(f"- {os.path.abspath(train_out_path)} ({df_train.shape[0]} dòng)")
print(f"- {os.path.abspath(test_out_path)} ({df_test.shape[0]} dòng)")

Đã lưu các file dữ liệu sạch thành công:
- t:\5 - Summer 2026\AES_LLM\data\processed\train.csv (8289 dòng)
- t:\5 - Summer 2026\AES_LLM\data\processed\test.csv (492 dòng)
